In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Mounted at /content/drive
/content/drive/MyDrive/Early-Sepsis-Detection


# 08 — Patient-Level Train/Validation/Test Split
### Early Sepsis Detection — Phase 10

Splits the 39,371-patient feature table (from `07_feature_engineering.ipynb`)
into train (70%) / validation (15%) / test (15%) sets, stratified by label,
with an explicit, mandatory verification that no patient appears in more
than one set.

**This split is locked here and reused unchanged by every later notebook**
(preprocessing, all models, tuning, threshold selection, calibration,
evaluation) — re-splitting differently in a later notebook would itself be
a leakage risk.


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
from src import split as sp

pd.set_option("display.max_columns", 60)


In [3]:
patient_df = pd.read_parquet(config.PROCESSED_PATIENT_LEVEL_PARQUET)
print("Loaded patient-level dataset:", patient_df.shape)
print("Label distribution:")
display(patient_df["label"].value_counts(normalize=True) * 100)


Loaded patient-level dataset: (39371, 467)
Label distribution:


,proportion
label,
0,93.744126
1,6.255874


## 1. Perform the stratified, patient-level split

Uses `config.RANDOM_SEED` for full reproducibility — running this notebook
again will always produce the exact same split.


In [4]:
splits = sp.patient_level_split(
    patient_df,
    train_frac=config.TRAIN_FRAC,
    val_frac=config.VAL_FRAC,
    test_frac=config.TEST_FRAC,
    seed=config.RANDOM_SEED,
)
for name, pids in splits.items():
    print(f"{name}: {len(pids)} patients")


2026-09-14 06:17:25,423 | INFO | src.split | Label=0: 36908 patients -> train=25836, val=5536, test=5536
INFO:src.split:Label=0: 36908 patients -> train=25836, val=5536, test=5536
2026-09-14 06:17:25,425 | INFO | src.split | Label=1: 2463 patients -> train=1724, val=369, test=370
INFO:src.split:Label=1: 2463 patients -> train=1724, val=369, test=370


train: 27560 patients
val: 5905 patients
test: 5906 patients


## 2. MANDATORY leakage check — verify pairwise intersections are empty

Per project requirements, this is not a courtesy check — training must not
proceed until all three values below are exactly 0.


In [5]:
overlaps = sp.verify_no_overlap(splits)
print(overlaps)

assert overlaps["train_val"] == 0, "LEAKAGE: train/val patient overlap detected!"
assert overlaps["train_test"] == 0, "LEAKAGE: train/test patient overlap detected!"
assert overlaps["val_test"] == 0, "LEAKAGE: val/test patient overlap detected!"
print("\nPASSED: train_patients ∩ val_patients = ∅, train_patients ∩ test_patients = ∅, "
      "val_patients ∩ test_patients = ∅")


2026-09-14 06:17:27,156 | INFO | src.split | Verified: train_val intersection is empty.
INFO:src.split:Verified: train_val intersection is empty.
2026-09-14 06:17:27,159 | INFO | src.split | Verified: train_test intersection is empty.
INFO:src.split:Verified: train_test intersection is empty.
2026-09-14 06:17:27,161 | INFO | src.split | Verified: val_test intersection is empty.
INFO:src.split:Verified: val_test intersection is empty.


{'train_val': 0, 'train_test': 0, 'val_test': 0}

PASSED: train_patients ∩ val_patients = ∅, train_patients ∩ test_patients = ∅, val_patients ∩ test_patients = ∅


## 3. Split summary — sizes and class balance

In [6]:
summary = sp.summarize_split(patient_df, splits)
display(summary)

# Sanity check: total patients across splits must equal the original count
total_split = summary["n_patients"].sum()
print(f"\nTotal patients across splits: {total_split} (original dataset: {len(patient_df)})")
assert total_split == len(patient_df), "Patient count mismatch after split!"


,split,n_patients,n_positive,n_negative,positive_pct
0,train,27560,1724,25836,6.255
1,val,5905,369,5536,6.249
2,test,5906,370,5536,6.265



Total patients across splits: 39371 (original dataset: 39371)


## 4. Save split assignments

Saved as a simple patient_id -> split lookup table so every later notebook
loads the SAME split rather than recomputing it (recomputing with even the
same seed in a different notebook risks subtle inconsistency if the
underlying patient_df order or filtering ever changes).


In [7]:
split_lookup = pd.concat([
    pd.DataFrame({config.PATIENT_ID_COL: pids, "split": name})
    for name, pids in splits.items()
], ignore_index=True)

split_lookup_path = config.PROCESSED_DIR / "patient_split_assignment.parquet"
split_lookup.to_parquet(split_lookup_path, index=False)
summary.to_csv(config.TABLES_DIR / "08_split_summary.csv", index=False)

print(f"Saved split assignments to {split_lookup_path}")
print(f"Saved split summary to reports/tables/08_split_summary.csv")
display(split_lookup.head())


Saved split assignments to /content/drive/MyDrive/Early-Sepsis-Detection/data/processed/patient_split_assignment.parquet
Saved split summary to reports/tables/08_split_summary.csv


,patient_id,split
0,p110358,train
1,p003396,train
2,p013874,train
3,p010999,train
4,p014913,train


---
### What to send back to Claude after running this notebook

- Section 2's overlap check output (must show all zeros — this is the gate
  before any modeling can proceed)
- Section 3's summary table (sizes and positive_pct per split — ideally
  all three splits show a similar ~6.26% prevalence)

With that, **Phase 10 is complete and verified**, and we move to
**Phase 11 (Preprocessing Pipeline)** — building a scikit-learn pipeline
(imputation + scaling) fitted ONLY on the training split, then
**Phase 12 (Class Imbalance Analysis)** before the first models in
Phase 13.
